# ⚖️ Libratio Fleet - Multi-Agent GPU Fleet Management
### Meta PyTorch OpenEnv Hackathon 2026

This notebook trains a tiny `Llama-3-8B` model using **GRPO (Reinforcement Learning)** to act as the Fleet Oversight Agent. It connects to our custom OpenEnv server simulating a multi-agent GPU cluster.

**Theme:** Multi-Agent Interactions + Fleet AI

In [ ]:
# 1. Install required packages (Unsloth, TRL, etc) for fast RL training
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install fastapi uvicorn httpx

In [ ]:
# 2. Clone your GitHub repository (Replace with your actual repo link once uploaded to GitHub)
# For now, since you are testing, you can upload your environment folders ('environment', 'scenarios', 'server') manually to the colab files area.
import os
import subprocess
import time

# Start the simulation server in the background of Colab
print("Starting Libratio Fleet Server...")
server_process = subprocess.Popen(["python", "-m", "uvicorn", "server.app:app", "--host", "127.0.0.1", "--port", "7860"])
time.sleep(5)  # Wait for it to boot
print("Server running!")

In [ ]:
# 3. Load the Model using Unsloth (Super fast!)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # Use 4-bit to fit in free Colab GPU
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Add LoRA adapters so it can learn our specific hackathon tasks
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Define the Reward Function (Connects to our server)
import httpx
import json

def fleet_oversight_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        # Extract the JSON action from the model's text output
        try:
            action_text = completion[0]['content']
            if "```json" in action_text:
                action_text = action_text.split("```json")[1].split("```")[0]
            action = json.loads(action_text)
            
            # Reset the environment and send the action to our FastAPI server
            httpx.post("http://127.0.0.1:7860/fleet/reset", json={"task_id": "fleet_oversight"})
            result = httpx.post("http://127.0.0.1:7860/fleet/step", json={"action": action}).json()
            
            # Give the score back to the RL trainer
            rewards.append(result["reward"]["score"])
        except Exception as e:
            rewards.append(0.01) # Penalize if it generates bad JSON
            
    return rewards

In [ ]:
# 5. Generate Training Data (What the model looks at to start)
from datasets import Dataset

# Ask our server for 100 random fleet scenarios
training_data = []
for _ in range(100):
    obs = httpx.post("http://127.0.0.1:7860/fleet/reset", json={"task_id": "fleet_oversight"}).json()["observation"]
    
    training_data.append({
        "prompt": [
            {"role": "system", "content": "You are a Fleet AI Oversight Agent monitoring multiple neural network training runs simultaneously. Output valid JSON."}, 
            {"role": "user", "content": f"Current environment state:\n{json.dumps(obs, default=str)}"}
        ]
    })

dataset = Dataset.from_list(training_data)

In [ ]:
# 6. Train the Agent using GRPO!
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="fleet_outputs",
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    max_steps=50,
    logging_steps=5,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[fleet_oversight_reward],
    args=training_args,
    train_dataset=dataset,
)

print("Starting Fleet RL Training...")
trainer.train()

In [ ]:
# 7. Save the trained agent!
model.save_pretrained("libratio_fleet_lora")
print("Model saved! Hackathon phase complete 🚀")